# CanOptimizer — Lagrange Multipliers in Multivariable Calculus (MTH-111)

**Institution:** Salim Habib University  
**Problem:** For a fixed volume $V_0$, find the radius $r$ and height $h$ of a closed cylindrical soda can that minimize total surface area:
$$f(r, h) = 2\pi r^2 + 2\pi r h \quad \text{subject to} \quad g(r, h) = \pi r^2 h = V_0.$$

This notebook runs the exact same SymPy symbolic derivation and SciPy SLSQP numerical verification used by the CanOptimizer web app (`lib/python/solver.py`).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import sympy as sp
from scipy.optimize import minimize

# Add project root so we can reuse lib/python/solver.py directly
sys.path.insert(0, str(Path('..').resolve() / 'lib' / 'python'))
from solver import solve_lagrange, verify_numerically, sweep_volume, sweep_weight

sp.init_printing()

## 1. Symbolic Lagrange Multiplier Derivation (SymPy)

In [ ]:
V0 = 355.0  # Standard 355 mL soda can (cm^3)
sol = solve_lagrange(V0)

print(f"Target Volume V0 = {sol['v0']} cm^3")
print(f"Optimal Radius r* = {sol['r_opt']:.6f} cm (Diameter d* = {sol['d_opt']:.6f} cm)")
print(f"Optimal Height h* = {sol['h_opt']:.6f} cm (h*/d* = {sol['h_over_d_opt']:.6f})")
print(f"Lagrange Multiplier lambda* = {sol['lambda_opt']:.6f} cm^-1")
print(f"Minimum Surface Area A_min = {sol['area_min']:.6f} cm^2")
print(f"Second Derivative A''(r*) = {sol['second_derivative_at_opt']:.4f} > 0 (Strict Minimum: {sol['is_strict_minimum']})")

## 2. Numerical Verification Across Multiple Starting Points & Volumes (SciPy SLSQP)

In [ ]:
for test_v0 in [250.0, 330.0, 355.0, 500.0, 1000.0]:
    ver = verify_numerically(test_v0, tolerance=1e-4)
    max_err = max(max(r['diff_r'], r['diff_h'], r['diff_area']) for r in ver['runs'])
    print(f"V0 = {test_v0:6.1f} cm^3 | All 6 SLSQP starts passed: {ver['all_passed']} | Max |diff| = {max_err:.2e}")

## 3. Interpreting $\lambda$ as Sensitivity $dA_{\min}/dV_0$

In [ ]:
sweep = sweep_volume(V0, v_min=100.0, v_max=1000.0)
print(f"At V0 = {V0} cm^3:")
print(f"  SymPy lambda*         = {sweep['lambda_at_v0']:.8f} cm^-1")
print(f"  Numerical dA_min/dV0  = {sweep['numerical_slope_at_v0']:.8f} cm^-1")
print(f"  Absolute Difference   = {sweep['abs_error']:.2e}")

## 4. Weighted Cost Extension ($f_k = 2\pi r h + k\cdot 2\pi r^2$)

In [ ]:
for k_val in [0.5, 1.0, 1.5, 2.0, 3.0]:
    s = solve_lagrange(V0, k_val=k_val)
    print(f"k = {k_val:3.1f} | r* = {s['r_opt']:.3f} cm | h* = {s['h_opt']:.3f} cm | h*/d* = {s['h_over_d_opt']:.3f}")